# บทที่ 08 - รูปแบบการออกแบบหลายตัวแทน


## การตั้งค่า


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ทำไมต้องใช้ระบบหลายตัวแทน?

งานในโลกจริงเช่นการวางแผนการเดินทางต้องใช้ความเชี่ยวชาญหลายด้าน — การจัดการโลจิสติกส์ ความรู้ท้องถิ่น งบประมาณ และอื่นๆ ตัวแทนเพียงหนึ่งเดียวที่พยายามจัดการทุกอย่างจะกลายเป็นสิ่งที่ยุ่งยากอย่างรวดเร็ว

ระบบหลายตัวแทนแก้ปัญหานี้ด้วยการ **เชี่ยวชาญเฉพาะทาง**: แต่ละตัวแทนจะมุ่งเน้นไปที่หนึ่งด้านของความเชี่ยวชาญ ทำให้ได้ผลลัพธ์ที่มีคุณภาพสูงกว่าผู้เชี่ยวชาญทั่วไป นอกจากนี้ยังช่วยเพิ่ม **ความสามารถในการปรับขนาด** — คุณสามารถเพิ่มตัวแทนใหม่ได้ (เช่น ผู้เชี่ยวชาญด้านเที่ยวบิน นักวิจารณ์ร้านอาหาร) โดยไม่ต้องเขียนโค้ดกระบวนการทำงานที่มีอยู่ใหม่ ตัวแทนจะถูกเชื่อมต่อกันผ่านท่อการทำงานที่มีโครงสร้าง ส่งต่อบริบทจากตัวหนึ่งไปยังตัวถัดไป


## การสร้างตัวแทนเฉพาะทาง


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## การสร้างลำดับงานแบบต่อเนื่อง

`WorkflowBuilder` ช่วยให้คุณเชื่อมต่อเอเจนต์เข้ากับกราฟที่มีทิศทาง ในที่นี้เราสร้างสายงานสองขั้นตอนง่ายๆ: **TravelPlanner** ร่างแผนการเดินทาง จากนั้น **TravelConcierge** ตรวจสอบและปรับปรุงแผนดังกล่าวต่อไป


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## การเพิ่มตัวแทนเพิ่มเติมในเวิร์กโฟลว์

หนึ่งในข้อดีที่ใหญ่ที่สุดของรูปแบบ multi-agent คือความง่ายในการขยาย ด้านล่างนี้เราจะเพิ่มตัวแทน **BudgetReviewer** ซึ่งตรวจสอบแผนกับงบประมาณของนักเดินทาง ทำเครื่องหมายรายการที่อาจทำให้ค่าใช้จ่ายเกินขีดจำกัด และแนะนำทางเลือกที่ช่วยประหยัดเงิน เวิร์กโฟลว์ตอนนี้จะรันตัวแทนสามตัวตามลำดับ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Summary

ในบทเรียนนี้คุณได้เรียนรู้วิธี:

1. **สร้างตัวแทนเฉพาะทาง** — แต่ละตัวมีบทบาทเฉพาะเจาะจง (การวางแผน, พนักงานต้อนรับ, การตรวจสอบงบประมาณ)
2. **เชื่อมต่อตัวแทนเข้ากับเวิร์กโฟลว์แบบลำดับ** โดยใช้ `WorkflowBuilder` และ `add_edge`
3. **สตรีมผลลัพธ์** จากสายการทำงานหลายตัวแทน โดยติดตามว่าตัวแทนใดกำลังพูด
4. **ขยายเวิร์กโฟลว์** โดยเพิ่มตัวแทนใหม่เข้าไปในสายงานโดยไม่แก้ไขตัวแทนที่มีอยู่

รูปแบบการออกแบบหลายตัวแทนช่วยให้แต่ละตัวแทนทำงานอย่างเรียบง่าย ในขณะที่สร้างผลลัพธ์ที่สมบูรณ์และผ่านการตรวจสอบอย่างละเอียดมากกว่าที่ตัวแทนเดียวจะทำได้เอง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:  
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาด้วย AI [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด ขอให้ท่านทราบว่าการแปลอัตโนมัติอาจมีความผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่มีความสำคัญ ขอแนะนำให้ใช้บริการแปลโดยมืออาชีพที่เป็นมนุษย์ เราจะไม่รับผิดชอบในกรณีที่เกิดความเข้าใจผิดหรือการตีความผิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
